## export consensus sequences for all motifs

- the goal here is to create a master dataframe of the motifs (to input in litemind queries)
- For each "motif", use the PFM to compute the "consensus" sequence, then add it to the motif dataframe.

- last updated: 7/9/2025

In [2]:
import pandas as pd
import numpy as np
# gimmemotifs

In [3]:
from gimmemotifs.motif import Motif, read_motifs

# Load motifs from the default database (all 5298 motifs)
motif_file = "/hpc/mydata/yang-joon.kim/genomes/danRer11/CisBP_ver2_Danio_rerio.pfm"  # Use the correct database file
motifs = read_motifs(motif_file)

In [4]:
print(f"the number of motifs is: {len(motifs)}")

the number of motifs is: 5298


### Step 1. Check the differentially enriched motifs for "coarse" and "fine" clustering

In [7]:
# quick check: compare the output motifs from the "leiden_coarse" and "leiden_unified"
# first, coarse: "leiden_coarse"
clust_by_motifs_coarse = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/leiden_by_motifs_maelstrom.csv",
                                     index_col=0)
clust_by_motifs_coarse.head()

# second, fine: ("leiden_unified")
clust_by_motifs_fine = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/leiden_unified_cluster_by_motifs.csv",
                                   index_col=0)
clust_by_motifs_fine.head()

,M08340_2.00,M03073_2.00,M06053_2.00,M04453_2.00,M11037_2.00,M04786_2.00,M02395_2.00,M08068_2.00,M03528_2.00,M09258_2.00,...,M06476_2.00,M10762_2.00,M06313_2.00,M02017_2.00,M04989_2.00,M07485_2.00,M02651_2.00,M08452_2.00,M10869_2.00,M06514_2.00
7_10,-0.915624,-0.116359,0.981625,0.321612,-3.356024,-1.603892,0.067387,0.509138,-2.367539,-2.080700,...,-0.097056,3.869513,1.888906,1.878376,-0.022904,-0.065015,0.695558,0.745282,2.222527,-0.654268
12_10,0.888473,-0.901515,1.188723,-0.337392,1.240034,1.524289,-0.599416,-0.540889,1.824949,0.240740,...,-1.625296,-0.328064,-0.654078,-1.622382,0.417666,-2.081952,-0.781643,-3.123044,-0.409468,-1.435741
33_3,2.819913,1.649425,0.133278,1.476795,1.060825,0.544359,-0.383817,1.460655,0.522318,1.820068,...,-1.060257,-1.269789,0.674509,-1.252817,-0.627832,-1.564414,-1.851004,-1.755666,-3.105792,-4.224889
26_9,1.174792,1.760438,0.067386,0.965038,1.446775,1.776728,-1.310191,1.109974,0.970842,2.661001,...,-1.673262,-0.009835,1.536371,-0.231220,-0.193513,-1.884204,-2.279405,-2.443068,-3.125721,-4.224889
33_6,2.953921,0.442935,-1.107456,0.154817,2.191585,0.695166,-1.068185,1.159113,1.108549,1.964671,...,-1.089171,-1.112487,1.013654,-1.085621,0.489869,-1.411336,-2.172929,-2.515371,-3.371039,-4.021660


In [12]:
# print the number of motifs that are unique in "coarse" or "fine" clusters from the maelstrom runs
motifs_unique_coarse = set(clust_by_motifs_coarse.columns) - set(clust_by_motifs_fine.columns)
motifs_unique_fine = set(clust_by_motifs_fine.columns) - set(clust_by_motifs_coarse.columns)

print(f"motifs in coarse clusters: {len(clust_by_motifs_coarse.columns)}")
print(f"motifs that are unique in coarse clusters: {len(motifs_unique_coarse)}")

print(f"motifs in fine clusters: {len(clust_by_motifs_fine.columns)}")
print(f"motifs that are unique in fine clusters: {len(motifs_unique_fine)}")

motifs in coarse clusters: 115
motifs that are unique in coarse clusters: 79
motifs in fine clusters: 118
motifs that are unique in fine clusters: 82


### Step 2. import the motif:TF dataframe for both "coarse" and "fine" clustering

#### 2-1. "coarse" motifs

In [13]:
# import the motifs post-gimme maelstrom
df_motif_info_coarse = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/maelstrom_640K_leiden_coarse_cisBP_v2_danio_rerio_output/info_cisBP_v2_danio_rerio_motif_factors.csv", index_col=0)
df_motif_info_coarse

,direct,indirect
motif,,
M00132_2.00,None,"myrf, osr2, osr1"
M00199_2.00,None,"sox19b, cica, sox5, sox11a, sox10, sox2, sox1b..."
M00210_2.00,None,"arid5b, foxg1b, sox2, foxa1, sox8a, sox7, sox4..."
M00216_2.00,None,"foxo4, irx7, foxg1b, foxo1b, barx1, dlx4a, hox..."
M00294_2.00,None,"hoxc13b, barhl1a, barx1, dlx4a, hoxb5b, hoxc11..."
...,...,...
M10805_2.00,None,"e2f3, barhl1a, barx1, dlx4a, hoxb5b, E2F1, bsx..."
M10816_2.00,None,"pax7a, pax2a, pax1a, pax3a, pax1b, pax8, pax6b..."
M10869_2.00,None,"hsf1, si:dkey-18a10.3, hsf4"


In [17]:
df_motif_info_coarse.index

Index(['M00132_2.00', 'M00199_2.00', 'M00210_2.00', 'M00216_2.00',
       'M00294_2.00', 'M00296_2.00', 'M00336_2.00', 'M00453_2.00',
       'M00539_2.00', 'M00621_2.00',
       ...
       'M10138_2.00', 'M10186_2.00', 'M10562_2.00', 'M10577_2.00',
       'M10784_2.00', 'M10805_2.00', 'M10816_2.00', 'M10869_2.00',
       'M10937_2.00', 'M11368_2.00'],
      dtype='object', name='motif', length=115)

In [27]:
# extract the consensus sequence for each motif,then create a list
list_consensus_seqs = []

for motif_name in df_motif_info_coarse.index:
    selected_motif = next(m for m in motifs if m.id == motif_name)

    # Extract the Position Frequency Matrix (PFM) and Position Weight Matrix (PWM)
    pfm = selected_motif.to_pfm()  # Transpose to make it compatible with logomaker
    pwm = selected_motif.to_ppm()

    # consensus sequence
    # print(motif_name)
    # print(selected_motif.to_consensus())
    # print(pwm)
    list_consensus_seqs.append(selected_motif.to_consensus())
    

# add the consensus sequence as additional column
df_motif_info_coarse["consensus"] = list_consensus_seqs

df_motif_info_coarse.head()

,direct,indirect,consensus
motif,,,
M00132_2.00,None,"myrf, osr2, osr1",nGCTACyGnn
M00199_2.00,None,"sox19b, cica, sox5, sox11a, sox10, sox2, sox1b...",nnnnwsAAT
M00210_2.00,None,"arid5b, foxg1b, sox2, foxa1, sox8a, sox7, sox4...",wTTGTnnnn
M00216_2.00,None,"foxo4, irx7, foxg1b, foxo1b, barx1, dlx4a, hox...",nwTwTAw
M00294_2.00,None,"hoxc13b, barhl1a, barx1, dlx4a, hoxb5b, hoxc11...",TyGTAAAA


In [24]:
# export the revised dataframe (motif for coarse)
df_motif_info_coarse.to_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/maelstrom_640K_leiden_coarse_cisBP_v2_danio_rerio_output/info_cisBP_v2_danio_rerio_motif_factors_consensus.csv")

#### 2-2. "fine" motifs

- We will have to import the MaelstromResult first, then export the dataframe of motifs:factors for the "fine" clusters (differentially enriched motifs among the fine clusters)

In [5]:
# import the motif:factors dataframe (from fine clusters)
df_motif_info_fine = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/maelstrom_640K_leiden_unified_cisBP_ver2_Danio_rerio_output/info_cisBP_v2_danio_rerio_motif_factors.csv",
                                 index_col=0)
df_motif_info_fine.head()

,direct,indirect
motif,,
M00199_2.00,None,"sox1b, sox11a, sox32, sox9a, sox8b, FQ377628.1..."
M00210_2.00,None,"sox32, sox9a, sox1b, sox11a, sox8b, FQ377628.1..."
M00216_2.00,None,"foxi2, mxtx2, zgc:162612, msx3, foxa3, evx2, o..."
M00238_2.00,None,"foxi2, nfyal, pbx3b, foxj3, foxk2, foxe1, gfi1..."
M00336_2.00,None,"E2F2, foxm1, e2f5, e2f3, pax6b, e2f4, e2f7, E2..."


In [6]:
# extract the consensus sequence for each motif,then create a list
list_consensus_seqs = []

for motif_name in df_motif_info_fine.index:
    selected_motif = next(m for m in motifs if m.id == motif_name)

    # Extract the Position Frequency Matrix (PFM) and Position Weight Matrix (PWM)
    pfm = selected_motif.to_pfm()  # Transpose to make it compatible with logomaker
    pwm = selected_motif.to_ppm()

    # consensus sequence
    # print(motif_name)
    # print(selected_motif.to_consensus())
    # print(pwm)
    list_consensus_seqs.append(selected_motif.to_consensus())
    

# add the consensus sequence as additional column
df_motif_info_fine["consensus"] = list_consensus_seqs

df_motif_info_fine.head()

,direct,indirect,consensus
motif,,,
M00199_2.00,None,"sox1b, sox11a, sox32, sox9a, sox8b, FQ377628.1...",nnnnwsAAT
M00210_2.00,None,"sox32, sox9a, sox1b, sox11a, sox8b, FQ377628.1...",wTTGTnnnn
M00216_2.00,None,"foxi2, mxtx2, zgc:162612, msx3, foxa3, evx2, o...",nwTwTAw
M00238_2.00,None,"foxi2, nfyal, pbx3b, foxj3, foxk2, foxe1, gfi1...",nAATCnnnn
M00336_2.00,None,"E2F2, foxm1, e2f5, e2f3, pax6b, e2f4, e2f7, E2...",GTGCACAC


In [7]:
df_motif_info_fine.to_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/maelstrom_640K_leiden_unified_cisBP_ver2_Danio_rerio_output/info_cisBP_v2_danio_rerio_motif_factors_fine_clusts_consensus.csv")

### Check the clusters-by-motifs dataframes

In [33]:
df_clusters_motifs = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/leiden_by_motifs_maelstrom.csv", index_col=0)
df_clusters_motifs.head()

,M04320_2.00,M04454_2.00,M01720_2.00,M10186_2.00,M03731_2.00,M04731_2.00,M04785_2.00,M02708_2.00,M00336_2.00,M04026_2.00,...,M06476_2.00,M02017_2.00,M06375_2.00,M08572_2.00,M06313_2.00,M02651_2.00,M10784_2.00,M08452_2.00,M10869_2.00,M06518_2.00
7,-1.368782,-0.369697,0.988747,1.306082,-1.572697,-1.347540,0.024850,1.216891,-0.962377,-0.956169,...,0.382885,1.929666,0.225133,2.097108,3.272975,1.766084,-2.053864,1.033707,1.907822,0.936034
12,-1.292497,-0.342407,0.079956,1.242187,-1.801436,-1.639597,0.062910,1.373438,-0.849191,-0.930442,...,0.780929,2.279517,0.482239,2.255553,3.281714,1.736472,-1.897626,0.559174,1.632902,0.612312
33,0.174508,0.025630,-0.238793,-1.375008,-1.139508,-0.960370,-0.658601,-1.154608,-2.564141,-0.832769,...,-1.672982,-0.038565,-2.422563,-0.989779,1.909328,-1.245577,-0.732930,-2.241918,-1.706974,-2.264252
26,0.115255,-0.303553,0.007131,-1.474846,-1.484947,0.433903,-0.152120,-1.225735,-2.716900,-0.382621,...,-1.601855,0.224392,-2.422563,-0.811669,2.232204,-1.245577,-0.564537,-2.241918,-1.706974,-2.264252
11,1.307492,1.226708,0.683540,-2.164871,-0.888929,1.245890,-0.437640,-2.335275,-2.373358,1.977257,...,-0.910050,-1.143654,-1.666987,-1.048229,0.789569,-1.861550,0.543802,-3.177755,-3.554546,-3.365767


In [34]:
df_fine_clusters_motifs = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/leiden_unified_cluster_by_motifs.csv", index_col=0)
df_fine_clusters_motifs.head()

,M08340_2.00,M03073_2.00,M06053_2.00,M04453_2.00,M11037_2.00,M04786_2.00,M02395_2.00,M08068_2.00,M03528_2.00,M09258_2.00,...,M06476_2.00,M10762_2.00,M06313_2.00,M02017_2.00,M04989_2.00,M07485_2.00,M02651_2.00,M08452_2.00,M10869_2.00,M06514_2.00
7_10,-0.915624,-0.116359,0.981625,0.321612,-3.356024,-1.603892,0.067387,0.509138,-2.367539,-2.080700,...,-0.097056,3.869513,1.888906,1.878376,-0.022904,-0.065015,0.695558,0.745282,2.222527,-0.654268
12_10,0.888473,-0.901515,1.188723,-0.337392,1.240034,1.524289,-0.599416,-0.540889,1.824949,0.240740,...,-1.625296,-0.328064,-0.654078,-1.622382,0.417666,-2.081952,-0.781643,-3.123044,-0.409468,-1.435741
33_3,2.819913,1.649425,0.133278,1.476795,1.060825,0.544359,-0.383817,1.460655,0.522318,1.820068,...,-1.060257,-1.269789,0.674509,-1.252817,-0.627832,-1.564414,-1.851004,-1.755666,-3.105792,-4.224889
26_9,1.174792,1.760438,0.067386,0.965038,1.446775,1.776728,-1.310191,1.109974,0.970842,2.661001,...,-1.673262,-0.009835,1.536371,-0.231220,-0.193513,-1.884204,-2.279405,-2.443068,-3.125721,-4.224889
33_6,2.953921,0.442935,-1.107456,0.154817,2.191585,0.695166,-1.068185,1.159113,1.108549,1.964671,...,-1.089171,-1.112487,1.013654,-1.085621,0.489869,-1.411336,-2.172929,-2.515371,-3.371039,-4.021660
